In [1]:
import os
import pandas as pd
import tkinter as tk
from tkinter import filedialog
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder

class ROS2BagMCAPToCSV:
    def __init__(self):
        self.decoder = Decoder()

    @staticmethod
    def flatten_dict(d, parent_key='', sep='.'):
        items = []
        # Convertimos a diccionario si es un objeto de ROS
        if hasattr(d, '__dict__'):
            d = vars(d)
        
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if hasattr(v, '__dict__') or isinstance(v, dict):
                items.extend(ROS2BagMCAPToCSV.flatten_dict(v, new_key, sep=sep).items())
            elif isinstance(v, list):
                items.append((new_key, str(v)))
            else:
                items.append((new_key, v))
        return dict(items)

    def process_single_file(self, mcap_path):
        """Procesa un solo archivo y guarda los CSVs en una carpeta dedicada."""
        output_dir = os.path.splitext(mcap_path)[0] + "_csv"
        os.makedirs(output_dir, exist_ok=True)
        
        data_by_topic = {}
        
        try:
            with open(mcap_path, "rb") as f:
                reader = make_reader(f)
                for schema, channel, message in reader.iter_messages():
                    topic_name = channel.topic
                    ros_msg = self.decoder.decode(schema, message)

                    # Extraer slots/atributos del mensaje
                    msg_dict = {slot: getattr(ros_msg, slot) for slot in dir(ros_msg) 
                                if not slot.startswith('_') and not callable(getattr(ros_msg, slot))}

                    record = {
                        "log_time_s": message.log_time / 1e9,
                        "publish_time_s": message.publish_time / 1e9,
                        **self.flatten_dict(msg_dict)
                    }

                    if topic_name not in data_by_topic:
                        data_by_topic[topic_name] = []
                    data_by_topic[topic_name].append(record)

            # Exportación a CSV
            for topic_name, records in data_by_topic.items():
                df = pd.DataFrame(records)
                clean_name = topic_name.strip("/").replace("/", "_")
                df.to_csv(os.path.join(output_dir, f"{clean_name}.csv"), index=False)
            
            print(f"✅ Procesado con éxito: {os.path.basename(mcap_path)}")
        except Exception as e:
            print(f"❌ Error procesando {mcap_path}: {e}")

    def run_recursive(self, root_folder):
        """Busca y procesa todos los .mcap en la carpeta y subcarpetas."""
        mcap_files = []
        for root, dirs, files in os.walk(root_folder):
            for file in files:
                if file.endswith(".mcap"):
                    mcap_files.append(os.path.join(root, file))
        
        if not mcap_files:
            print("No se encontraron archivos .mcap en el directorio seleccionado.")
            return

        print(f"🔍 Se encontraron {len(mcap_files)} archivos. Iniciando conversión...")
        for path in mcap_files:
            print(f"🚀 Procesando: {path}")
            self.process_single_file(path)
        print("\n✨ ¡Proceso finalizado!")

def main():
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    folder_path = filedialog.askdirectory(title="Selecciona la carpeta raíz para buscar MCAPs")

    if folder_path:
        converter = ROS2BagMCAPToCSV()
        converter.run_recursive(folder_path)
    else:
        print("Operación cancelada.")

if __name__ == '__main__':
    main()

🔍 Se encontraron 6 archivos. Iniciando conversión...
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251124_094548\20251124_094548_0.mcap
✅ Procesado con éxito: 20251124_094548_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_111941\20251217_111941_0.mcap
✅ Procesado con éxito: 20251217_111941_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_120944\20251217_120944_0.mcap
✅ Procesado con éxito: 20251217_120944_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_125205\20251217_125205_0.mcap
✅ Procesado con éxito: 20251217_125205_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_125816\20251217_125816_0.mcap
✅ Procesado con éxito: 20251217_125816_0.mcap
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag\20251217_131